In [ ]:
import os, re, time, numpy as np, pandas as pd

SEED, PER_CLASS, VAL_FRAC = 42, 4000, 0.15
CLASSES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
CLS2IDX = {c: i for i, c in enumerate(CLASSES)}
root = '/kaggle/input/datasets/paultimothymooney/kermany2018/oct2017/OCT2017 '

rows = []
for split in ('train', 'test'):
    for c in CLASSES:
        d = f'{root}/{split}/{c}'
        if not os.path.isdir(d):
            continue
        for fn in os.listdir(d):
            m = re.match(r'([A-Za-z]+)-(\d+)-(\d+)', fn)
            if m:
                rows.append((f'{d}/{fn}', split, m.group(1).upper(), m.group(2), fn))

df = pd.DataFrame(rows, columns=['path', 'split', 'cls', 'pid', 'fname'])
df = df.drop_duplicates(['split', 'fname']).reset_index(drop=True)
df['label'] = df.cls.map(CLS2IDX)
assert df.label.notna().all(), "Unexpected class name"

train_full = df[df.split == 'train']
test_df = df[df.split == 'test'].reset_index(drop=True)

shared = set(train_full.pid) & set(test_df.pid)
same_cls = set(zip(train_full.cls, train_full.pid)) & set(zip(test_df.cls, test_df.pid))
print("Shared patient IDs train/test:", len(shared), "| same (class, ID) pairs:", len(same_cls))
print("Train images from those patients (excluded):", int(train_full.pid.isin(shared).sum()))

pool = train_full[~train_full.pid.isin(shared)]
print("Pool after exclusion:", pool.cls.value_counts().to_dict())

rng = np.random.RandomState(SEED)
pids = pool.pid.unique(); rng.shuffle(pids)
val_p = set(pids[:int(len(pids) * VAL_FRAC)])
tr_pool, va_pool = pool[~pool.pid.isin(val_p)], pool[pool.pid.isin(val_p)]

n_val = int(PER_CLASS * VAL_FRAC); n_tr = PER_CLASS - n_val
def cap(d, n):
    return pd.concat([d[d.cls == c].sample(min(n, (d.cls == c).sum()), random_state=SEED) for c in CLASSES])

tr = cap(tr_pool, n_tr).sample(frac=1, random_state=SEED).reset_index(drop=True)
va = cap(va_pool, n_val).sample(frac=1, random_state=SEED).reset_index(drop=True)

assert not set(tr.pid) & set(va.pid)
assert not (set(tr.pid) | set(va.pid)) & set(test_df.pid)
print("PATIENT OVERLAP CHECK PASSED")
print("Train:", tr.cls.value_counts().to_dict())
print("Val:  ", va.cls.value_counts().to_dict())
print("Test: ", len(test_df))

tr.to_csv('/kaggle/working/train_manifest.csv', index=False)
va.to_csv('/kaggle/working/val_manifest.csv', index=False)
test_df.to_csv('/kaggle/working/test_manifest.csv', index=False)
print("Saved manifests.")

In [ ]:
assert 'tr_pool' in globals(), "Rerun the previous split cell first"
CAP = None   # None = use ALL train images; or e.g. 10000 per class for faster training

def cap(d, n):
    if n is None:
        return d
    return pd.concat([d[d.cls == c].sample(min(n, (d.cls == c).sum()), random_state=SEED) for c in CLASSES])

tr = cap(tr_pool, CAP).sample(frac=1, random_state=SEED).reset_index(drop=True)
va = va_pool.sample(frac=1, random_state=SEED).reset_index(drop=True)

assert not set(tr.pid) & set(va.pid)
assert not (set(tr.pid) | set(va.pid)) & set(test_df.pid)
print("PATIENT OVERLAP CHECK PASSED")
print("Train:", tr.cls.value_counts().to_dict(), "| total", len(tr))
print("Val:  ", va.cls.value_counts().to_dict(), "| total", len(va))
cnt = tr.cls.value_counts()
print("Class weights:", {c: round(len(tr) / (4 * cnt[c]), 2) for c in CLASSES})

tr.to_csv('/kaggle/working/train_manifest.csv', index=False)
va.to_csv('/kaggle/working/val_manifest.csv', index=False)
test_df.to_csv('/kaggle/working/test_manifest.csv', index=False)
print("Saved manifests.")

In [ ]:
import numpy as np, pandas as pd, cv2, time
from concurrent.futures import ThreadPoolExecutor
cv2.setNumThreads(1)

def preprocess_img(gray):
    g = cv2.GaussianBlur(gray, (0, 0), 1)
    g = cv2.medianBlur(g, 3)
    return cv2.resize(g, (224, 224), interpolation=cv2.INTER_AREA)

def load(p):
    return preprocess_img(cv2.imread(p, cv2.IMREAD_GRAYSCALE))

def build(name):
    df = pd.read_csv(f'/kaggle/working/{name}_manifest.csv'); t = time.time(); imgs = []
    with ThreadPoolExecutor(16) as ex:
        for i, im in enumerate(ex.map(load, df['path'])):
            imgs.append(im)
            if i % 5000 == 0: print(name, i, '/', len(df), f'{time.time()-t:.0f}s')
    np.save(f'/kaggle/working/{name}_x.npy', np.stack(imgs))
    np.save(f'/kaggle/working/{name}_y.npy', df['label'].values.astype(np.int64))
    print(name, 'done', len(df), f'{time.time()-t:.0f}s')

for n in ('val', 'test', 'train'):
    build(n)

#### import numpy as np, pandas as pd, cv2, time
from concurrent.futures import ThreadPoolExecutor
cv2.setNumThreads(1)

def load(p):
    return preprocess_img(cv2.imread(p, cv2.IMREAD_GRAYSCALE))

def build(name):
    df = pd.read_csv(f'/kaggle/working/{name}_manifest.csv'); t = time.time(); imgs = []
    with ThreadPoolExecutor(16) as ex:
        for i, im in enumerate(ex.map(load, df['path'])):
            imgs.append(im)
            if i % 5000 == 0: print(name, i, '/', len(df), f'{time.time()-t:.0f}s')
    np.save(f'/kaggle/working/{name}_x.npy', np.stack(imgs))
    np.save(f'/kaggle/working/{name}_y.npy', df['label'].values.astype(np.int64))
    print(name, 'done', len(df), f'{time.time()-t:.0f}s')

for n in ('val', 'test', 'train'):
    build(n)

In [ ]:
!ls -lh /kaggle/working/*.npy

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, torchvision, numpy as np, time, math
from sklearn.metrics import f1_score

dev, W = 'cuda', '/kaggle/working'
CLASSES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
Xtr = torch.from_numpy(np.load(f'{W}/train_x.npy')); ytr = torch.from_numpy(np.load(f'{W}/train_y.npy'))
Xva = torch.from_numpy(np.load(f'{W}/val_x.npy'));   yva = torch.from_numpy(np.load(f'{W}/val_y.npy'))
mean = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)

def prep(xb, train=False):
    x = xb.to(dev, non_blocking=True).float().div(255).unsqueeze(1)
    if train:                                   # random flip + rotation up to 10 degrees, on GPU
        n = x.size(0)
        f = (torch.rand(n, device=dev) < 0.5).float() * 2 - 1
        a = (torch.rand(n, device=dev) * 2 - 1) * (10 * math.pi / 180)
        c, s = torch.cos(a), torch.sin(a)
        th = torch.zeros(n, 2, 3, device=dev)
        th[:, 0, 0], th[:, 0, 1], th[:, 1, 0], th[:, 1, 1] = c * f, -s, s * f, c
        x = F.grid_sample(x, F.affine_grid(th, x.shape, align_corners=False),
                          padding_mode='reflection', align_corners=False)
    return (x.repeat(1, 3, 1, 1) - mean) / std

model = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(2048, 4); model = model.to(dev)
cnt = np.bincount(ytr.numpy(), minlength=4)
lossf = nn.CrossEntropyLoss(weight=torch.tensor(len(ytr) / (4 * cnt), dtype=torch.float32, device=dev))
scaler = torch.amp.GradScaler('cuda')
BS, best, gep = 64, 0.0, 0

def evaluate():
    model.eval(); P = []
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
        for i in range(0, len(Xva), 256):
            P.append(model(prep(Xva[i:i + 256])).argmax(1).cpu())
    p, y = torch.cat(P).numpy(), yva.numpy()
    return f1_score(y, p, average='macro'), (p == y).mean()

def fit(n_ep, lr, params, tag, freeze_bn):
    global best, gep
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    spe = len(Xtr) // BS
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_ep * spe, eta_min=1e-6)
    for ep in range(n_ep):
        model.train()
        if freeze_bn:
            for m in model.modules():
                if isinstance(m, nn.BatchNorm2d): m.eval()
        perm, t, tl, k = torch.randperm(len(Xtr)), time.time(), 0.0, 0
        for b in range(spe):
            idx = perm[b * BS:(b + 1) * BS]
            x, y = prep(Xtr[idx], True), ytr[idx].to(dev)
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.float16):
                loss = lossf(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            if b % 50 == 0: tl += loss.item(); k += 1
        f1, acc = evaluate(); gep += 1
        ck = {'state': model.state_dict(), 'classes': CLASSES, 'epoch': gep, 'val_f1': f1}
        torch.save(ck, f'{W}/ckpt_ep{gep}.pt')
        if f1 > best:
            best = f1; torch.save(ck, f'{W}/best_model.pt')
        print(f'[{tag}] ep {gep} | loss {tl/k:.3f} | val macro-F1 {f1:.4f} | acc {acc:.4f} | {time.time()-t:.0f}s')

for p in model.parameters(): p.requires_grad = False
for p in model.fc.parameters(): p.requires_grad = True
fit(3, 1e-3, model.fc.parameters(), 'head', True)         # stage 1: head only, 3 epochs
for p in model.parameters(): p.requires_grad = True
fit(10, 1e-4, model.parameters(), 'full', False)          # stage 2: full network, 10 epochs
print('BEST val macro-F1:', round(best, 4))

In [ ]:
import torch, torch.nn as nn, torchvision, numpy as np, json
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay, precision_recall_fscore_support

dev, W = 'cuda', '/kaggle/working'
CLASSES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
ck = torch.load(f'{W}/best_model.pt', map_location=dev)
print('Best checkpoint: epoch', ck['epoch'], '| val macro-F1', round(ck['val_f1'], 4))
model = torchvision.models.resnet50(weights=None); model.fc = nn.Linear(2048, 4)
model.load_state_dict(ck['state']); model = model.to(dev).eval()
mean = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)

def predict(X):
    out = []
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
        for i in range(0, len(X), 256):
            x = torch.from_numpy(X[i:i + 256]).to(dev).float().div(255).unsqueeze(1).repeat(1, 3, 1, 1)
            out.append(torch.softmax(model((x - mean) / std).float(), 1).cpu())
    return torch.cat(out).numpy()

Xva, yva = np.load(f'{W}/val_x.npy'), np.load(f'{W}/val_y.npy')
pv = predict(Xva); pred = pv.argmax(1); conf = pv.max(1)
print(classification_report(yva, pred, target_names=CLASSES, digits=3))
print(confusion_matrix(yva, pred))

print('\nthreshold | kept % | accuracy on kept')
ts, ok = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95], []
for t in ts:
    keep = conf >= t; acc = (pred[keep] == yva[keep]).mean()
    print(f'{t:.2f}      | {keep.mean()*100:5.1f}  | {acc*100:.2f}')
    if acc >= 0.98: ok.append(t)
THR = ok[0] if ok else 0.9
json.dump({'threshold': THR}, open(f'{W}/threshold.json', 'w'))
print('\nChosen threshold (smallest with >=98% accuracy on kept, else 0.9):', THR)

In [ ]:
Xte, yte = np.load(f'{W}/test_x.npy'), np.load(f'{W}/test_y.npy')
pt = predict(Xte); pred = pt.argmax(1); conf = pt.max(1)
np.save(f'{W}/test_probs.npy', pt)
print(classification_report(yte, pred, target_names=CLASSES, digits=3))
p, r, f, _ = precision_recall_fscore_support(yte, pred, labels=[0, 1, 2, 3])
print(f'Accuracy {(pred == yte).mean():.4f} | Macro-F1 {f.mean():.4f}')
print(f'DME: precision {p[1]:.3f} | recall {r[1]:.3f} | F1 {f[1]:.3f}')
cm = confusion_matrix(yte, pred)
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(cmap='Blues'); plt.title('Test confusion matrix')
plt.savefig(f'{W}/confusion_matrix.png', dpi=150, bbox_inches='tight'); plt.show()
THR = json.load(open(f'{W}/threshold.json'))['threshold']
unc = conf < THR
print(f'Threshold {THR}: flagged uncertain {unc.mean()*100:.1f}% | accuracy on the rest {(pred[~unc] == yte[~unc]).mean()*100:.2f}%')

In [ ]:
!pip install -q grad-cam
import os, cv2
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
os.makedirs(f'{W}/gradcam', exist_ok=True)
cam = GradCAM(model=model, target_layers=[model.layer4[-1]])

def save(i, tag):
    g = Xte[i]
    x = (torch.from_numpy(g).float().div(255)[None, None].repeat(1, 3, 1, 1).to(dev) - mean) / std
    heat = cam(input_tensor=x, targets=None)[0]
    rgb = np.repeat((g / 255.0).astype(np.float32)[..., None], 3, axis=2)
    im = np.hstack([(rgb * 255).astype(np.uint8), show_cam_on_image(rgb, heat, use_rgb=True)])
    cv2.imwrite(f'{W}/gradcam/{tag}.jpg', cv2.cvtColor(im, cv2.COLOR_RGB2BGR))

for c in range(4):                                   # one confident correct case per class
    idx = np.where((yte == c) & (pred == c))[0]
    if len(idx):
        j = idx[np.argmax(conf[idx])]; save(j, f'correct_{CLASSES[c]}_conf{conf[j]:.2f}')
wrong = np.where(pred != yte)[0]                     # most confident failures
for i in wrong[np.argsort(-conf[wrong])][:4]:
    save(i, f'wrong_true-{CLASSES[yte[i]]}_pred-{CLASSES[pred[i]]}_conf{conf[i]:.2f}')

fs = sorted(os.listdir(f'{W}/gradcam'))
fig, ax = plt.subplots(4, 2, figsize=(10, 10))
for a, f_ in zip(ax.flat, fs):
    a.imshow(cv2.imread(f'{W}/gradcam/{f_}')[..., ::-1]); a.set_title(f_, fontsize=7); a.axis('off')
plt.show()